# 316 — Orchestrator & Interactive Chat

The main user-facing demo notebook.

An ipywidgets chat interface routes natural-language questions through the
multi-agent pipeline (Orchestrator → ComplianceAgent + InvestigationAgent
→ AnswerSynthesisAgent) and displays structured results with:

- Routing decision
- Generated Cypher (expandable)
- Answer
- Evidence panel (cited sections + chunks)
- Anomaly findings panel
- Recommended next steps

**Pre-loaded example questions** use real entity IDs from the data.

In [ ]:
%run 311_agent_setup.ipynb

In [ ]:
from src.agent.orchestrator import Orchestrator
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

orchestrator = Orchestrator(tools=TOOLS, execute_tool_fn=execute_tool)
print('Orchestrator ready.')

## Chat UI

In [ ]:
# ── Colour scheme ────────────────────────────────────────────────────────────
COLOURS = {
    'routing':    '#e8f5e9',
    'cypher':     '#f5f5f5',
    'answer':     '#e3f2fd',
    'evidence':   '#fff9c4',
    'anomaly':    '#fce4ec',
    'steps':      '#f3e5f5',
    'user_msg':   '#e0e0e0',
}
SEV_COLOUR = {'HIGH': '#f8d7da', 'MEDIUM': '#fff3cd', 'LOW': '#d4edda', 'INFO': '#d1ecf1'}

# ── Panel renderers ───────────────────────────────────────────────────────────

def _panel(title: str, content: str, colour: str) -> widgets.HTML:
    return widgets.HTML(f"""
    <div style="border-left:4px solid #888;padding:8px 12px;margin:4px 0;
                background:{colour};border-radius:4px;font-size:13px">
      <b style="font-size:11px;text-transform:uppercase;color:#555">{title}</b>
      <div style="margin-top:4px">{content}</div>
    </div>""")

def _render_routing(routing: dict) -> str:
    intents  = ', '.join(routing.get('intents', []))
    entities = ', '.join(routing.get('entity_ids', []) or ['—'])
    regs     = ', '.join(routing.get('regulations', []) or ['—'])
    agents   = []
    if routing.get('needs_compliance_agent'): agents.append('ComplianceAgent')
    if routing.get('needs_investigation_agent'): agents.append('InvestigationAgent')
    if routing.get('run_anomaly_check'): agents.append('AnomalyCheck')
    return (f'<b>Intent:</b> {intents} &nbsp;|&nbsp; '
            f'<b>Entities:</b> {entities} &nbsp;|&nbsp; '
            f'<b>Regs:</b> {regs}<br>'
            f'<b>Agents:</b> {" → ".join(agents) or "—"}')

def _render_cypher(cypher_list: list) -> widgets.Accordion:
    items = []
    for i, c in enumerate(cypher_list):
        q = c.get('cypher', c) if isinstance(c, dict) else c
        items.append(widgets.HTML(f'<pre style="font-size:11px;white-space:pre-wrap">{q}</pre>'))
    acc = widgets.Accordion(children=items)
    for i in range(len(items)):
        acc.set_title(i, f'Cypher query {i+1}')
    acc.selected_index = None
    return acc

def _render_findings(findings: list) -> str:
    if not findings:
        return '<i>No findings.</i>'
    lines = []
    for f in findings:
        sev = f.get('severity', 'INFO')
        c   = SEV_COLOUR.get(sev, '#fff')
        lines.append(
            f'<div style="background:{c};padding:4px 8px;margin:2px 0;border-radius:3px">'
            f'<b>[{sev}]</b> {f.get("description", "")}</div>'
        )
    return ''.join(lines)

def _render_evidence(resp) -> str:
    parts = []
    if resp.cited_sections:
        parts.append('<b>Cited sections:</b>')
        for s in resp.cited_sections:
            parts.append(f'<div style="margin-left:12px">{s.get("section_id")} — {s.get("title", "")}</div>')
    if resp.cited_chunks:
        parts.append('<b>Cited chunks:</b>')
        for c in resp.cited_chunks:
            excerpt = (c.get('text_excerpt') or '')[:200]
            parts.append(
                f'<div style="margin-left:12px"><code>{c.get("chunk_id")}</code> '
                f'(score: {c.get("similarity_score", "")})<br>'
                f'<i>{excerpt}…</i></div>'
            )
    return ''.join(parts) if parts else '<i>No evidence cited.</i>'

def render_response(resp) -> list:
    """Return a list of widgets representing the full response."""
    wids = []

    # Verdict badge
    vc = {'COMPLIANT': '#28a745', 'NON_COMPLIANT': '#dc3545',
          'REQUIRES_REVIEW': '#fd7e14', 'ANOMALY_DETECTED': '#dc3545',
          'INFORMATIONAL': '#6c757d'}.get(resp.verdict, '#6c757d')
    wids.append(widgets.HTML(
        f'<span style="background:{vc};color:white;padding:3px 10px;'
        f'border-radius:12px;font-size:12px;font-weight:bold">{resp.verdict}</span>'
        f'&nbsp;<span style="color:#888;font-size:12px">confidence: {resp.confidence:.0%}</span>'
    ))

    # Routing
    wids.append(_panel('Routing', _render_routing(resp.routing), COLOURS['routing']))

    # Cypher (collapsed accordion)
    if resp.cypher_used:
        wids.append(widgets.HTML('<b style="font-size:11px;color:#555">CYPHER USED</b>'))
        wids.append(_render_cypher(resp.cypher_used))

    # Answer
    answer_html = resp.answer.replace('\n', '<br>')
    wids.append(_panel('Answer', answer_html, COLOURS['answer']))

    # Evidence
    if resp.cited_sections or resp.cited_chunks:
        wids.append(_panel('Evidence', _render_evidence(resp), COLOURS['evidence']))

    # Findings
    if resp.findings:
        wids.append(_panel('Findings', _render_findings(resp.findings), COLOURS['anomaly']))

    # Next steps
    if resp.recommended_next_steps:
        steps_html = '<ol>' + ''.join(f'<li>{s}</li>' for s in resp.recommended_next_steps) + '</ol>'
        wids.append(_panel('Recommended Next Steps', steps_html, COLOURS['steps']))

    # Assessment ID link
    if resp.assessment_id:
        wids.append(widgets.HTML(
            f'<span style="font-size:11px;color:#888">'
            f'Assessment stored: <code>{resp.assessment_id}</code></span>'
        ))

    return wids

print('Renderers ready.')

In [ ]:
# ── Chat state ────────────────────────────────────────────────────────────────
history: list[dict] = []

# ── Widgets ───────────────────────────────────────────────────────────────────
question_box = widgets.Textarea(
    placeholder='Ask a compliance or investigation question…',
    layout=widgets.Layout(width='100%', height='72px'),
)
ask_btn   = widgets.Button(description='Ask',   button_style='primary', icon='search')
clear_btn = widgets.Button(description='Clear', button_style='warning', icon='trash')
status    = widgets.HTML('')
out       = widgets.Output()

# Example question buttons
EXAMPLES = [
    'Is LOAN-0002 compliant with APG-223?',
    'Show suspicious connections around BRW-0001.',
    'Find all transaction structuring patterns.',
    'Show the ownership chain behind BRW-0582.',
    'Why might LOAN-0013 require manual review?',
    'Which APRA thresholds apply to residential secured loans?',
]
example_btns = [
    widgets.Button(description=q[:55], layout=widgets.Layout(width='100%', margin='1px 0'),
                   button_style='info')
    for q in EXAMPLES
]

def _redraw():
    with out:
        clear_output(wait=True)
        for h in history:
            if h['role'] == 'user':
                display(widgets.HTML(
                    f'<div style="background:{COLOURS["user_msg"]};padding:8px 12px;'
                    f'border-radius:4px;margin:4px 0"><b>You:</b> {h["content"]}</div>'
                ))
            else:
                for w in h['widgets']:
                    display(w)
                display(widgets.HTML('<hr style="margin:8px 0">'))

def on_ask(b):
    q = question_box.value.strip()
    if not q:
        return
    question_box.value = ''
    history.append({'role': 'user', 'content': q})

    status.value = '<i style="color:#888">⏳ Thinking…</i>'
    _redraw()

    try:
        resp = orchestrator.run(q)
        history.append({'role': 'assistant', 'widgets': render_response(resp)})
    except Exception as e:
        history.append({'role': 'assistant', 'widgets': [
            widgets.HTML(f'<div style="color:red">Error: {e}</div>')
        ]})

    status.value = ''
    _redraw()

def on_clear(b):
    history.clear()
    with out:
        clear_output()

def make_example_handler(question: str):
    def handler(b):
        question_box.value = question
    return handler

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)
for btn, q in zip(example_btns, EXAMPLES):
    btn.on_click(make_example_handler(q))

# ── Layout ────────────────────────────────────────────────────────────────────
examples_box = widgets.VBox(
    [widgets.HTML('<b style="font-size:11px;color:#555">EXAMPLE QUESTIONS</b>')] + example_btns,
    layout=widgets.Layout(width='280px', margin='0 8px 0 0'),
)
input_area = widgets.VBox([
    widgets.HTML('<b style="font-size:13px">🔍 Graph Investigation Assistant</b>'),
    question_box,
    widgets.HBox([ask_btn, clear_btn, status]),
])
top_row   = widgets.HBox([examples_box, input_area])
full_ui   = widgets.VBox([top_row, out])

display(full_ui)

---
## Direct API usage (no UI)

For scripted testing or integration:

In [ ]:
# Run a question directly and inspect the structured response
resp = orchestrator.run('Find all transaction structuring patterns.')
print(f'Verdict:    {resp.verdict}')
print(f'Confidence: {resp.confidence}')
print(f'Findings:   {len(resp.findings)}')
print(f'Cypher:     {len(resp.cypher_used)} queries')
print(f'\nAnswer:\n{resp.answer[:500]}')